# Data Loading and Preprocessing

This notebook is the single entry point for the creativity-ratings data.
It downloads the current rating export from Google Sheets, optionally
attaches the matching StatsBomb event details from the local event
database, and then applies every step that turns the raw export into an
analysis-ready dataset: schema checks, type conversion, out-of-range and
duplicate handling, and exclusion of ratings where the action could not
be recognized.

Both `01_creativity_ratings_python.ipynb` and `01_creativity_ratings_R.qmd`
read the result of this notebook (`data/processed/ratings_long_clean.csv`)
directly and do not repeat any of these steps themselves. Running this
notebook is therefore a prerequisite for both of the language-specific
analysis notebooks. `01_data_overview.ipynb` also reads the same file for
a quick, non-technical look at the data before the full analysis is run.

Two files are produced:

- `data/raw/ratings_long_raw.csv` — the sheet export, saved immediately
  and never modified afterwards.
- `data/processed/ratings_long_clean.csv` — the cleaned, analysis-ready
  dataset.

Every row removed or changed along the way is recorded in
`results/tables/cleaning_log.csv`, together with the affected rows saved
separately under `results/tables/flagged_*.csv`.

## 1. Setup

In [1]:
import warnings
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Project paths (relative to the notebook folder "notebooks/")
# ------------------------------------------------------------------
PROJECT_ROOT = Path("..").resolve()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RESULTS_TABLES = PROJECT_ROOT / "results" / "tables"

for d in [DATA_RAW, DATA_PROCESSED, RESULTS_TABLES]:
    d.mkdir(parents=True, exist_ok=True)

# The shared StatsBomb event database lives one level above this project,
# alongside the other numbered project folders (00-data, 01-rating-study, ...).
EVENT_DB_PATH = PROJECT_ROOT.parents[2] / "00-data" / "statsbomb_event_data.duckdb"

def save_table(df_table, name, index=False):
    # Saves a table as CSV under results/tables/.
    path = RESULTS_TABLES / f"{name}.csv"
    df_table.to_csv(path, index=index)
    print(f"Table saved: {path.relative_to(PROJECT_ROOT)} "
          f"({df_table.shape[0]} rows x {df_table.shape[1]} columns)")
    return path

WARNINGS_LOG = []
def log_warning(msg):
    WARNINGS_LOG.append({"timestamp": datetime.now(timezone.utc).isoformat(),
                          "message": msg})
    warnings.warn(msg)

cleaning_log = []
def log_clean_step(step, n_before, n_after, description):
    cleaning_log.append({
        "step": step, "n_rows_before": n_before, "n_rows_after": n_after,
        "n_rows_affected": n_before - n_after, "description": description,
    })

print(f"Project root: {PROJECT_ROOT}")

Project root: /home/max/drive/projects/3_Creativity/11-paper-CAT-methodology/data-analysis/creativity-ratings-psychometrics


## 2. Load ratings from Google Sheets

The rating export is a shared Google Sheet, downloaded here via its
public XLSX export URL (the sheet is shared as "anyone with the link");
no authentication is required.

In [2]:
sheet_url = "https://docs.google.com/spreadsheets/d/1KUcLAxm5g-zz9myrTcQOLH5L09JmFCRaIJ0woCzY1TA/edit?usp=sharing"
sheet_id = sheet_url.split("/d/")[1].split("/")[0]
url_template = "https://docs.google.com/spreadsheets/d/{}/export?format=xlsx"

ratings_raw = pd.read_excel(url_template.format(sheet_id), sheet_name=0)
print(f"Downloaded {len(ratings_raw)} rows.")
print(f"Columns: {list(ratings_raw.columns)}")
ratings_raw.head()

Downloaded 8750 rows.
Columns: ['user_id', 'id', 'action_not_recognized', 'creativity_rating', 'technical_correctness', 'aesthetic_appeal', 'device_type', 'os', 'os_version', 'browser', 'browser_version', 'user_agent', 'timestamp']


,user_id,id,action_not_recognized,creativity_rating,technical_correctness,aesthetic_appeal,device_type,os,os_version,browser,browser_version,user_agent,timestamp
0,urst3254,0786bb62-2032-444b-ac1f-59fa165dbb71,False,1.0,2.0,2.0,local,NaN,NaN,NaN,NaN,NaN,2026-03-11 11:50:29.717
1,anma2416,f90addfa-4cf8-4b07-a4eb-9ee7ff757519,False,2.0,4.0,1.0,local,NaN,NaN,NaN,NaN,NaN,2025-11-18 12:26:15.275
2,anma2416,2bdc8133-ac12-45f1-9104-882873efa7bf,False,3.0,4.0,2.0,local,NaN,NaN,NaN,NaN,NaN,2025-10-15 13:35:18.000
3,azne2115,938d2c93-5e6f-442e-ba7c-f92c53080a42,False,5.0,5.0,5.0,local,NaN,NaN,NaN,NaN,NaN,2026-05-05 15:58:08.508
4,urst3254,4fe7a32b-09ce-466b-9772-494ed03ea493,False,6.0,7.0,7.0,local,NaN,NaN,NaN,NaN,NaN,2026-04-08 13:56:06.665


In [3]:
# Save the raw data immediately after downloading it, and do NOT modify
# this file afterwards.
RAW_PATH = DATA_RAW / "ratings_long_raw.csv"
ratings_raw.to_csv(RAW_PATH, index=False)
print(f"Raw data saved: {RAW_PATH} ({len(ratings_raw)} rows)")

Raw data saved: /home/max/drive/projects/3_Creativity/11-paper-CAT-methodology/data-analysis/creativity-ratings-psychometrics/data/raw/ratings_long_raw.csv (8750 rows)


## 3. Event details from the StatsBomb event database (optional)

Each rated action corresponds to a StatsBomb event. If the local event
database is reachable, we look up the matching events here and save them
as a separate table, keyed by action id, alongside the ratings. This is
supplementary context (action type, players, teams, ...); it is not
required by — and not merged into — the ratings used for the
psychometric analysis.

In [4]:
try:
    import duckdb

    if not EVENT_DB_PATH.exists():
        raise FileNotFoundError(EVENT_DB_PATH)

    action_ids = ratings_raw["id"].unique().tolist()
    id_list = "', '".join(str(a) for a in action_ids)

    conn = duckdb.connect(str(EVENT_DB_PATH))
    action_events_raw = conn.execute(
        f"SELECT * FROM events WHERE id IN ('{id_list}')"
    ).fetchdf()
    conn.close()

    EVENTS_PATH = DATA_RAW / "action_events_raw.csv"
    action_events_raw.to_csv(EVENTS_PATH, index=False)
    print(f"Event details saved: {EVENTS_PATH} ({len(action_events_raw)} actions)")
except Exception as exc:
    log_warning(
        f"Could not load event details from {EVENT_DB_PATH} ({exc}). "
        "Continuing without event data - it is optional and not used "
        "by the psychometric analysis."
    )

Event details saved: /home/max/drive/projects/3_Creativity/11-paper-CAT-methodology/data-analysis/creativity-ratings-psychometrics/data/raw/action_events_raw.csv (4106 actions)


## 4. Map columns to the analysis schema

The exported sheet uses its own column names. We map them onto the
schema the analysis notebooks expect (`action_id`, `rater_id`, `rating`),
keep the two additional rating dimensions the sheet also collects
(`technical_correctness`, `aesthetic_appeal`) as unused extra columns for
possible future analyses, and drop purely technical export metadata
(device/OS/browser) that has no analytical use here — it remains
available in the raw file if ever needed.

In [5]:
df = ratings_raw.rename(columns={
    "id": "action_id",
    "user_id": "rater_id",
    "creativity_rating": "rating",
}).copy()

df["action_id"] = df["action_id"].astype(str)
df["rater_id"] = df["rater_id"].astype(str)

for extra_col in ["technical_correctness", "aesthetic_appeal"]:
    if extra_col in df.columns:
        df[extra_col] = pd.to_numeric(df[extra_col], errors="coerce")

metadata_cols = ["device_type", "os", "os_version", "browser", "browser_version", "user_agent"]
df = df.drop(columns=[c for c in metadata_cols if c in df.columns])

print(f"Columns after mapping: {list(df.columns)}")

Columns after mapping: ['rater_id', 'action_id', 'action_not_recognized', 'rating', 'technical_correctness', 'aesthetic_appeal', 'timestamp']


## 5. Exclude ratings where the action was not recognized

Raters could flag an action as not recognizable (`action_not_recognized`).
Those rows carry no real creativity judgment, so they are excluded from
the analyzed dataset here rather than passed downstream. As with every
other exclusion in this notebook, the affected rows are saved separately
and the exclusion is recorded in the cleaning log.

In [6]:
n0 = len(df)

if "action_not_recognized" in df.columns:
    not_recognized_mask = df["action_not_recognized"].fillna(False).astype(bool)
    tbl_not_recognized = df.loc[not_recognized_mask]
    print(f"Rows flagged as action_not_recognized: {len(tbl_not_recognized)}")
    if len(tbl_not_recognized) > 0:
        save_table(tbl_not_recognized, "flagged_action_not_recognized")
        log_warning(f"{len(tbl_not_recognized)} rows excluded: action_not_recognized was set.")

    n_before = len(df)
    df = df.loc[~not_recognized_mask].drop(columns=["action_not_recognized"]).copy()
    log_clean_step("drop_action_not_recognized", n_before, len(df),
                    "Rows where the rater flagged the action as not recognized were excluded")
else:
    log_warning("No action_not_recognized column found - skipping this exclusion step.")

Rows flagged as action_not_recognized: 1248
Table saved: results/tables/flagged_action_not_recognized.csv (1248 rows x 7 columns)


/tmp/ipykernel_390163/1055534179.py:35: UserWarning: 1248 rows excluded: action_not_recognized was set.
  warnings.warn(msg)


## 6. Schema check and type conversion

We check that the required variables exist, that IDs are not missing,
that `rating` is numeric or safely convertible to numeric, and that
values fall within the allowed range \(\{1, \dots, 7\}\).

In [7]:
required_cols = ["action_id", "rater_id", "rating"]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Required variables missing from the raw dataset: {missing_cols}")

n0 = len(df)

# --- missing IDs ------------------------------------------------------
n_missing_action_id = df["action_id"].isna().sum()
n_missing_rater_id = df["rater_id"].isna().sum()
print(f"Missing action_id: {n_missing_action_id}")
print(f"Missing rater_id:  {n_missing_rater_id}")
if n_missing_action_id or n_missing_rater_id:
    log_warning(f"{n_missing_action_id + n_missing_rater_id} rows with a missing ID found.")
n_before = len(df)
df = df.dropna(subset=["action_id", "rater_id"]).copy()
log_clean_step("drop_missing_id", n_before, len(df), "Rows with missing action_id/rater_id removed")

# --- convert rating to numeric (robust against " 4 ", "n/a", etc.) -----
rating_numeric_raw = pd.to_numeric(
    df["rating"].astype(str).str.strip(), errors="coerce"
)
n_not_convertible = rating_numeric_raw.isna().sum() - df["rating"].isna().sum()
print(f"Non-convertible, non-missing raw rating values: {n_not_convertible}")
if n_not_convertible > 0:
    print(df.loc[rating_numeric_raw.isna() & df["rating"].notna(),
                  ["action_id", "rater_id", "rating"]])
    log_warning(f"{n_not_convertible} raw rating values were not numerically convertible and were set to NA.")
df["rating_numeric_raw"] = rating_numeric_raw

Missing action_id: 0
Missing rater_id:  0
Non-convertible, non-missing raw rating values: 0


In [8]:
# --- check value range: only 1-7 allowed -------------------------------
out_of_range_mask = df["rating_numeric_raw"].notna() & (
    (df["rating_numeric_raw"] < 1) | (df["rating_numeric_raw"] > 7)
    | (df["rating_numeric_raw"] % 1 != 0)
)
tbl_out_of_range = df.loc[out_of_range_mask,
                            ["action_id", "rater_id", "rating", "rating_numeric_raw"]]
print(f"Values outside {{1,...,7}} or non-integer: {len(tbl_out_of_range)}")
if len(tbl_out_of_range) > 0:
    print(tbl_out_of_range)
    log_warning(f"{len(tbl_out_of_range)} rating values outside the allowed range 1-7 found and removed.")
    save_table(tbl_out_of_range, "flagged_out_of_range_ratings")

n_before = len(df)
df = df.loc[~out_of_range_mask].copy()
log_clean_step("drop_out_of_range", n_before, len(df),
                "Rating values outside {1,...,7} or non-integer removed")

# --- missing ratings (after conversion) --------------------------------
n_before = len(df)
n_missing_rating = df["rating_numeric_raw"].isna().sum()
print(f"Missing/non-convertible rating values: {n_missing_rating}")
if n_missing_rating > 0:
    log_warning(f"{n_missing_rating} rows with missing/non-convertible rating removed.")
df = df.loc[df["rating_numeric_raw"].notna()].copy()
log_clean_step("drop_missing_rating", n_before, len(df),
                "Rows with missing or non-convertible rating removed")

Values outside {1,...,7} or non-integer: 1
                                 action_id  rater_id  rating  \
4301  0980706d-7a40-4dc9-88dd-61e669af79a2  urst3254     0.0   

      rating_numeric_raw  
4301                 0.0  
Table saved: results/tables/flagged_out_of_range_ratings.csv (1 rows x 4 columns)
Missing/non-convertible rating values: 0


/tmp/ipykernel_390163/1055534179.py:35: UserWarning: 1 rating values outside the allowed range 1-7 found and removed.
  warnings.warn(msg)


## 7. Duplicate (`action_id` x `rater_id`) combinations

**Documented resolution rule:** repeated occurrences of the same
rater-action combination are first listed in full in a table. Since
there is no reliable way to tell which of the repeated entries is the
intended one, we apply an explicit, conservative rule: *the row that
appears last in the sheet is kept* (a later entry in the export
typically reflects a correction). This assumption should be revisited
if the actual cause of duplicates in the sheet becomes known.

In [9]:
dup_mask = df.duplicated(subset=["action_id", "rater_id"], keep=False)
tbl_duplicates = df.loc[dup_mask,
                          ["action_id", "rater_id", "rating", "rating_numeric_raw"]].sort_values(
    ["action_id", "rater_id"])
print(f"Rows with a duplicate (action_id, rater_id) combination: {len(tbl_duplicates)}")
if len(tbl_duplicates) > 0:
    print(tbl_duplicates)
    save_table(tbl_duplicates, "flagged_duplicate_rater_action_pairs")
    log_warning(
        f"{len(tbl_duplicates)} rows with duplicate (action_id, rater_id) "
        "combinations found. Resolution rule: the last entry is kept "
        "(see the markdown cell above)."
    )

n_before = len(df)
df = df.drop_duplicates(subset=["action_id", "rater_id"], keep="last").copy()
log_clean_step("resolve_duplicates", n_before, len(df),
                "Duplicates resolved: last entry per (action_id, rater_id) kept")

# --- final rating variables --------------------------------------------
df["rating_numeric"] = df["rating_numeric_raw"].astype(int)
df["rating_ordinal"] = pd.Categorical(df["rating_numeric"], categories=list(range(1, 8)),
                                        ordered=True)
df = df.drop(columns=["rating_numeric_raw"])

assert df["rating_numeric"].between(1, 7).all(), "Ratings outside 1-7 after cleaning!"
assert not df.duplicated(subset=["action_id", "rater_id"]).any(), "Duplicates remain after cleaning!"
assert df["action_id"].notna().all() and df["rater_id"].notna().all()

print(f"\nCleaning complete: {n0} -> {len(df)} rows.")

Rows with a duplicate (action_id, rater_id) combination: 12
                                 action_id  rater_id  rating  \
7796  0a9c032f-abc4-4769-a5a7-c1a7bf349592    CNNW38     1.0   
7797  0a9c032f-abc4-4769-a5a7-c1a7bf349592    CNNW38     1.0   
7953  20bb4549-e015-4cba-9b1f-603b05173d56    ZVNQ65     1.0   
7955  20bb4549-e015-4cba-9b1f-603b05173d56    ZVNQ65     1.0   
8256  2541577f-83c0-42ae-9d71-e70411f210e2    CNNW38     3.0   
8257  2541577f-83c0-42ae-9d71-e70411f210e2    CNNW38     3.0   
8298  28ee9cab-bac5-4b8f-94da-d1f788534f94    CNNW38     4.0   
8299  28ee9cab-bac5-4b8f-94da-d1f788534f94    CNNW38     4.0   
8267  2e17dd71-a4ba-4289-8c49-a66baf0594dd    CNNW38     1.0   
8268  2e17dd71-a4ba-4289-8c49-a66baf0594dd    CNNW38     1.0   
3850  36a6bfe9-af61-4c28-b27f-205c9706b4cc  nayü2116     2.0   
7614  36a6bfe9-af61-4c28-b27f-205c9706b4cc  nayü2116     1.0   

      rating_numeric_raw  
7796                 1.0  
7797                 1.0  
7953                 1.0  

/tmp/ipykernel_390163/1055534179.py:35: UserWarning: 12 rows with duplicate (action_id, rater_id) combinations found. Resolution rule: the last entry is kept (see the markdown cell above).
  warnings.warn(msg)


## 8. Rating order within each rater's session

The sheet records a `timestamp` for every rating but not an explicit
`rating_order`. We derive `rating_order` by ranking each rater's ratings
chronologically, so the temporal-drift check in the analysis notebooks
(which looks for a `rating_order` column) can run on the real data.

In [10]:
if "timestamp" in df.columns:
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df["rating_order"] = (
        df.sort_values("timestamp")
        .groupby("rater_id")
        .cumcount() + 1
    )
    df = df.sort_index()
else:
    log_warning("No timestamp column found - rating_order could not be derived.")

## 9. Save the preprocessed data

This is the file both analysis notebooks (`01_creativity_ratings_python.ipynb`,
`01_creativity_ratings_R.qmd`) and the overview notebook
(`01_data_overview.ipynb`) read as their starting point.

In [11]:
CLEAN_PATH = DATA_PROCESSED / "ratings_long_clean.csv"
df.to_csv(CLEAN_PATH, index=False)
print(f"Cleaned data saved: {CLEAN_PATH} ({len(df)} rows)")

cleaning_log_df = pd.DataFrame(cleaning_log)
save_table(cleaning_log_df, "cleaning_log")
print(cleaning_log_df)

print(f"\nRaters: {df['rater_id'].nunique()}")
print(f"Actions: {df['action_id'].nunique()}")
print(f"Warnings collected: {len(WARNINGS_LOG)}")
df.head()

Cleaned data saved: /home/max/drive/projects/3_Creativity/11-paper-CAT-methodology/data-analysis/creativity-ratings-psychometrics/data/processed/ratings_long_clean.csv (7495 rows)
Table saved: results/tables/cleaning_log.csv (5 rows x 5 columns)
                         step  n_rows_before  n_rows_after  n_rows_affected  \
0  drop_action_not_recognized           8750          7502             1248   
1             drop_missing_id           7502          7502                0   
2           drop_out_of_range           7502          7501                1   
3         drop_missing_rating           7501          7501                0   
4          resolve_duplicates           7501          7495                6   

                                         description  
0  Rows where the rater flagged the action as not...  
1       Rows with missing action_id/rater_id removed  
2  Rating values outside {1,...,7} or non-integer...  
3  Rows with missing or non-convertible rating re...  
4  D

,rater_id,action_id,rating,technical_correctness,aesthetic_appeal,timestamp,rating_numeric,rating_ordinal,rating_order
0,urst3254,0786bb62-2032-444b-ac1f-59fa165dbb71,1.0,2.0,2.0,2026-03-11 11:50:29.717,1,1,5
1,anma2416,f90addfa-4cf8-4b07-a4eb-9ee7ff757519,2.0,4.0,1.0,2025-11-18 12:26:15.275,2,2,708
2,anma2416,2bdc8133-ac12-45f1-9104-882873efa7bf,3.0,4.0,2.0,2025-10-15 13:35:18.000,3,3,17
3,azne2115,938d2c93-5e6f-442e-ba7c-f92c53080a42,5.0,5.0,5.0,2026-05-05 15:58:08.508,5,5,6
4,urst3254,4fe7a32b-09ce-466b-9772-494ed03ea493,6.0,7.0,7.0,2026-04-08 13:56:06.665,6,6,403
